In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1" 
os.environ['TORCH_USE_CUDA_DSA'] = "1"

# 🎯 Bitirme Projesi: Duygu Bias'lı LLM Decoder Katmanları ile Doğal Duygulu Cevap Üretimi

## 1. Giriş
- **Problem Tanımı:** LLM'ler genellikle nötr cevaplar üretir. Amaç, decoder katmanlarına eklenen ağırlıklı duygu bias'ları ile doğal duygulu cevaplar üretmek.
- **Motivasyon:** İnsan-makine etkileşiminde duygusal tonun önemi.
- **Literatür Özeti:** Controlled text generation, sentiment control, prefix-tuning, bias injection teknikleri.

---

## 2. Teorik Arka Plan
- **LLM Mimarisi:** Encoder-Decoder vs Decoder-only modeller.
- **Decoder Katman Yapısı:** Self-attention, feed-forward, layer norm, bias noktaları.
- **Duygu Temsili:** 
  - One-hot veya soft-weighted vektörler (ör. `[happy:0.6, sad:0.2, angry:0.2]`)
  - Embedding dönüşümü ile bias vektörüne çevrilmesi.
- **Bias Enjeksiyon Noktaları:**
  - Katman girişine ekleme
  - Attention skorlarına ekleme
  - Output logits’e ekleme

---

## 3. Yöntem
### 3.1 Mimari Tasarım
- **Mini Encoder:** Duygu ağırlık listesini alıp bias embedding üreten küçük bir ağ.
- **Bias Enjeksiyonu:** Decoder katmanlarının her birine eklenmesi.
- **Formül:**
  $
  h'_l = h_l + W_b \cdot e_{emotion}
  $
  Burada \( $h_l$ \) katman aktivasyonu, \( $e_{emotion}$ \) duygu embedding’i, \( $W_b$ \) öğrenilebilir ağırlık matrisi.

### 3.2 Veri Seti
- **Temel veri:** Nötr diyalog veri seti (örn. DailyDialog, OpenSubtitles).
- **Ek etiketleme:** Duygu etiketleri (happy, sad, angry, neutral).

### 3.3 Eğitim Stratejisi
- **Fine-tuning:** Mevcut LLM üzerine duygu bias modülü ekleyerek.
- **Loss Fonksiyonu:**
  - Ana dil modelleme loss’u (CrossEntropy)
  - Duygu uyum loss’u (örn. cosine similarity ile hedef duygu embedding’e yakınlık)

---

## 4. Deneysel Kurulum
- **Model:** Küçük boyutlu bir GPT-2 / OPT / LLaMA tabanlı decoder.
- **Ortam:** PyTorch + HuggingFace Transformers.
- **Hyperparametreler:** batch size, learning rate, bias boyutu.

---

## 5. Uygulama Adımları (Kod Hücreleri)
Prompt → Hafif Encoder (duygu çıkarımı) → Bias Encoder (d_model boyutunda vektör)
       → GPT-benzeri Decoder (her katmana bias enjeksiyonu) → Çıktı

# Faz 1 (Duygu analizi için fine-tunning yapılır)

In [ ]:
# venv\Scripts\activate 
# Faz 1: Hafif Encoder Eğitimi
# ============================
os.environ["HF_HOME"] = os.path.join(os.getcwd(), ".hf_cache")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(os.getcwd(), ".hf_cache", "transformers")
os.environ["HF_DATASETS_CACHE"] = os.path.join(os.getcwd(), ".hf_cache", "datasets")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer
from copy import deepcopy
from transformers import GPT2LMHeadModel, AutoTokenizer
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, GPT2LMHeadModel
from transformers.cache_utils import DynamicCache

In [ ]:
# 1) Veri seti yükleme (GoEmotions) - cache proje klasöründe
dataset = load_dataset("go_emotions", "simplified", cache_dir="./.hf_cache", download_mode="reuse_dataset_if_exists")  # 27 emotion + neutral
num_emotions = len(dataset["train"].features["labels"].feature.names)

# 2) Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="./.hf_cache")

# 3) Tokenize + labels'ı one-hot encode et
def tokenize_and_encode(batch):
    # Tokenize
    tokenized = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
    # One-hot encode labels
    labels = torch.zeros((len(batch["labels"]), num_emotions), dtype=torch.float)
    for i, label_list in enumerate(batch["labels"]):
        labels[i, label_list] = 1.0
    tokenized["labels"] = labels
    return tokenized

dataset = dataset.map(tokenize_and_encode, batched=True)

# 4) PyTorch formatına çevir
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

"""
input_ids:     [1012, 4567,    0,    0]
attention_mask:[   1,    1,    0,    0]
"""

# 5) DataLoader'lar
train_loader = DataLoader(dataset["train"], batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(dataset["validation"], batch_size=8)

In [ ]:
# 3) Model tanımı
class EmotionExtractor(nn.Module):
    def __init__(self, encoder_name, num_emotions):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_emotions)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionExtractor(model_name, num_emotions).to(device)

# 4) Loss ve optimizer
loss_fn = nn.BCEWithLogitsLoss() # çıktıda sigmoid koymadık çünkü BCEWithLogitsLoss bu zaten otomatik loss hesabında log alıyor zaten.
optimizer = AdamW(model.parameters(), lr=5e-5)

if os.path.exists("checkpoint.pt"):
    print("Checkpoint bulundu, kaldığı yerden devam ediliyor...")
    checkpoint = torch.load("checkpoint.pt", map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
else:
    print("Checkpoint bulunamadı, sıfırdan başlanıyor...")
    if os.path.exists("emotion_extractor.pt"):
        print("Önceden eğitilmiş model ağırlıkları yükleniyor...")
        model.load_state_dict(torch.load("emotion_extractor.pt", map_location=device))
    # optimizer zaten sıfırdan başlıyor


In [ ]:
# 5) Eğitim döngüsü
for epoch in range(3):
    model.train()
    torch.set_grad_enabled(True)
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].float().to(device)
        
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f}")
    
    # Validation
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.extend((probs > 0.5).astype(int))
            trues.extend(labels)
    f1 = f1_score(trues, preds, average="micro")
    print(f"Validation micro-F1: {f1:.4f}")

# 6) Modeli kaydet
    # 6) Model + optimizer + epoch + loss kaydet
    torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': total_loss/len(train_loader)
    }, "checkpoint.pt")
    

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

model.eval()
preds, trues = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()
        
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        preds.extend((probs > 0.5).astype(int))
        trues.extend(labels)

# Precision, Recall, F1 (micro/macro/weighted)
precision_micro = precision_score(trues, preds, average="micro")
recall_micro    = recall_score(trues, preds, average="micro")
f1_micro        = f1_score(trues, preds, average="micro")

precision_macro = precision_score(trues, preds, average="macro")
recall_macro    = recall_score(trues, preds, average="macro")
f1_macro        = f1_score(trues, preds, average="macro")

print(f"Micro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1:        {f1_micro:.4f}")
print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1:        {f1_macro:.4f}")

# Ayrıntılı rapor (her sınıf için)
print(classification_report(trues, preds, target_names=[str(i) for i in range(num_emotions)]))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

# GoEmotions dataset'i yükle (simplified versiyon için subset parametresi kullanılabilir)
dataset = load_dataset("go_emotions", "simplified")

# Label isimlerini çek
emotion_labels = dataset['train'].features['labels'].feature.names
print(emotion_labels)  # ['admiration','amusement','anger',...,'neutral']

# Class-wise metrikler (senin çıktından alınan değerler)
precision = [0.68,0.75,0.51,0.43,0.46,0.46,0.61,0.51,0.76,0.45,
             0.42,0.59,0.79,0.33,0.74,0.92,0.50,0.66,0.75,0.67,
             0.68,1.00,0.47,0.20,0.68,0.70,0.58,0.74]
recall    = [0.78,0.84,0.38,0.26,0.26,0.41,0.27,0.56,0.42,0.09,
             0.42,0.34,0.31,0.22,0.47,0.90,0.08,0.48,0.75,0.38,
             0.52,0.33,0.17,0.06,0.69,0.36,0.48,0.46]
f1        = [0.73,0.79,0.43,0.33,0.33,0.43,0.37,0.54,0.54,0.15,
             0.42,0.43,0.45,0.26,0.57,0.91,0.13,0.56,0.75,0.48,
             0.59,0.50,0.24,0.09,0.69,0.48,0.53,0.56]

x = np.arange(len(emotion_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(16,6))
ax.bar(x - width, precision, width, label='Precision')
ax.bar(x, recall, width, label='Recall')
ax.bar(x + width, f1, width, label='F1')

ax.set_xlabel('Emotion Class')
ax.set_ylabel('Score')
ax.set_title('Class-wise Precision, Recall, F1')
ax.set_xticks(x)
ax.set_xticklabels(emotion_labels, rotation=90)  # sınıf isimlerini yaz
ax.legend()

plt.tight_layout()
plt.savefig("classwise_metrics.png", dpi=300)
plt.show()